# **Building a Retrieval-Augmented Generation (RAG) System**

In this lecture, we will walk through the process of building a **Retrieval-Augmented Generation (RAG)** system.

### **Use Case**

We will be building a **RAG-based customer support assistant** for **Ready Tensor** the company behind this AI course. Our goal is to create a system that can answer user questions about the platform by retrieving relevant information from its documentation.

### **Tech Stack Overview**

* **Embedding Model**: `all-MiniLM-L6-v2` — a lightweight and efficient transformer model from Hugging Face used to convert text into vector embeddings.
* **Vector Database**: `ChromaDB` — a fast and persistent vector store used to store and retrieve document chunks based on similarity.
* **Language Model (LLM)**: `Groq` - used to synthesize human-like responses based on the retrieved context.

### **What You'll Learn**

By the end of this lecture, you will:

* Understand how to preprocess and chunk raw documentation.
* Embed and store documents in a vector database for fast retrieval.
* Implement a retriever that finds the most relevant document chunks in response to user queries.
* Use the retrieved content to generate accurate, grounded responses using a language model.



In [ ]:
%pip install langchain-groq langchain-community langchain chromadb sentence-transformers langchain-huggingface

In [1]:
import os
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

If you remember in week 2 when we talked about RAG system, we said there are two parts. 
1. Insertion Part
2. Retrieval Part

Now, we are going to build the retrieval part. Below are the steps we are going to follow:
1. Load the documents
2. Split them into chunks
3. Create embeddings for each chunk 
4. Store the embeddings in our vector database (ChromaDB)

## 1. Load the documents from the documents folder

In [4]:
# Path to the documents folder
documents_path = "../documents"
    
# List to store all documents
documents = []
    
# Load each .txt file in the documents folder
for file in os.listdir(documents_path):
    if file.endswith(".txt"):
        file_path = os.path.join(documents_path, file)
        try:
            loader = TextLoader(file_path)
            documents.extend(loader.load())
            print(f"Successfully loaded: {file}")
        except Exception as e:
            print(f"Error loading {file}: {str(e)}")
    
print(f"\nTotal documents in the documents loaded: {len(documents)}")

Successfully loaded: ready_tensor_kb_part1.txt
Successfully loaded: ready_tensor_kb_part3.txt
Successfully loaded: ready_tensor_kb_part2.txt

Total documents in the documents loaded: 3


## 2. Split documents into chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

print(f"Split into {len(splits)} chunks")

Split into 14 chunks


## 3. Embed the chunks using any embedding model of your choice
- Below we have 4 options to choose from (all of them are open source embedding models)
- We will use `all-MiniLM-L6-v2` embedding model for this example
- You can comment out the one you want to use and use it

In [ ]:
# Option 1: ChromaDB's Default Embedding (Recommended for simplicity)
# from chromadb.utils import embedding_functions

# # Use ChromaDB's default embedding function
# default_ef = embedding_functions.DefaultEmbeddingFunction()

# Option 2: Sentence Transformers (from huggingface)
from langchain_huggingface import HuggingFaceEmbeddings

sentence_transformers_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# # Option 3: OpenAI-like Embeddings
# from langchain_community.embeddings import OllamaEmbeddings

# ollama_embeddings = OllamaEmbeddings(
#     model="nomic-embed-text"  # An open-source embedding model just like OpenAI's embedding model
# )

# # Option 4: Other Popular Open-Source Models
# other_embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-large-en-v1.5"  # Another high-quality embedding model
# )


# Choose your embedding model
embeddings = sentence_transformers_embeddings

## 4. Store the embeddings in our vector database (ChromaDB)

In [9]:
# Create and persist the vector store
vectorstore = Chroma.from_documents(
    documents=splits, # the chunks we split from the documents
    embedding=embeddings, # the embedding model we chose
    persist_directory="./chroma_db" # the directory to save the vector store
)

print("Vector store created and documents stored!")

Vector store created and documents stored!


Now that we’ve successfully completed the **insertion stage** where we loaded our data, chunked it using `RecursiveCharacterTextSplitter`, embedded it using the `all-MiniLM-L6-v2` model from Hugging Face, and stored it in ChromaDB we move to the **retrieval stage** of our Retrieval-Augmented Generation (RAG) system.

In this stage, we will:

1. Accept a user’s query.
2. Embed the query using the same embedding model (`all-MiniLM-L6-v2`).
3. Use the embedded query to perform a similarity search against our Chroma vector store.
4. Retrieve the most relevant document chunks based on cosine similarity.
5. Finally, use groq Model to modify the relevant document so it can have a user friendly response.

Let's get started with it


## Set up the environment variables

You'll need a Groq API key to make calls to their models. If you don't have one yet, you can sign up at [https://console.groq.com](https://console.groq.com) and create a free API key.

In [10]:
import os
os.environ["GROQ_API_KEY"] = "put your groq api key here"  

## Initialize the Groq chat model

In [11]:
llm = ChatGroq(
    model="llama3-70b-8192",  # You can also use "mixtral-8x7b-32768" or other models
    temperature=0.1,
    max_tokens=1024
)

In [13]:
def rag_query(query):
    """
    Perform a Retrieval-Augmented Generation (RAG) query on the vector store.
    
    Args:
        query (str): The user's input query
    
    Returns:
        str: A user-friendly response based on the most relevant document chunks
    """
    
    # Create a retriever (
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # k is the number of chunks to retrieve from the vector store 
    
    # Create a prompt template (system message) for the RAG system
    prompt_template = ChatPromptTemplate.from_template("""You are a knowledgeable and friendly customer support assistant for Ready Tensor — an AI collaboration platform for publishing articles and managing datasets, models, workflows, and research.

    Use the following context from Ready Tensor's documentation to answer the user's question as accurately and helpfully as possible.

    If the answer is not explicitly found in the context, respond politely that you're unsure and suggest checking the official Ready Tensor documentation or contacting support.

    Context:
    {context}

    User Question:
    {input}

    Answer as a Ready Tensor support assistant:""")

    
    # Create a document chain
    document_chain = create_stuff_documents_chain(
        llm, 
        prompt_template
    )
    
    # Create a retrieval chain
    retrieval_chain = create_retrieval_chain(
        retriever, 
        document_chain
    )
    
    # Run the retrieval chain
    response = retrieval_chain.invoke({
        "input": query
    })
    
    # Return the answer
    return response['answer']


## Test the RAG system with a sample query

In [14]:
query = "What is Ready Tensor all about?"
result = rag_query(query)
print(result)

Hello there!

Ready Tensor is an AI collaboration platform that focuses on openness and responsibility. At its core, it enables the sharing of datasets, models, workflows, and research through its Publications System. This system is supported by robust tools and competitions, with a strong emphasis on security, reproducibility, and community-driven quality assessment.

In simpler terms, Ready Tensor is a collaborative space where researchers, developers, and scientists can come together to work on AI-related projects, share knowledge, and build upon each other's work. We provide pre-trained models for domain-specific tasks, learning materials such as tutorials and webinars, and community forums for Q&A.

Our goal is to create a platform that fosters innovation, transparency, and accountability in AI research and development. If you have any more questions or would like to know more, feel free to ask!


In [15]:
query = "What features does Ready Tensor offer?"
result = rag_query(query)
print(result)

Hello there!

Ready Tensor offers a range of exciting features that facilitate collaborative AI development and research. Our platform is designed with openness, responsibility, and community-driven quality assessment at its core.

Some of the key features include:

1. **Publication System**: This allows users to share and collaborate on various types of publications, including datasets, models, workflows, and research papers.

2. **Competition Support**: Our platform enables the creation of competitions, which fosters a spirit of innovation and collaboration among users.

3. **Robust Tools**: We provide a suite of tools that support various aspects of AI development, such as data preprocessing, model training, and workflow automation.

4. **Security and Reproducibility**: We prioritize security and reproducibility are central to our platform's design, ensuring that users can trust the integrity of their work and collaborations.

5. **Community-driven Quality Assessment**: Our communit

In [16]:
query = "Who can use Ready Tensor?"
result = rag_query(query)
print(result)

Hello there!

Ready Tensor is designed to be an open and inclusive platform, which means that anyone can use it! Whether you're a researcher, scientist, engineer, student, or simply someone interested in AI and collaboration, you're welcome to join our community and utilize our platform.

Our platform is particularly useful for those involved in end-to-end AI development, and research, as we provide a range of tools and resources to support your work. From hyperparameter tuning to data preprocessing and visualization, we've got you covered.

So, feel free to sign up and start exploring our platform today! If you have any questions or need help getting started, don't hesitate to reach out. We're always here to assist you.


Let's assume you don't want to use an LLM to answer the user's question yet, you just want to see the chunks being retrieved from the vector store.
You can run the following code to see the chunks being retrieved from the vector store.

In [17]:
# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # k is the number of chunks to retrieve from the vector store 
    
# Perform similarity search to retrieve chunks
retrieved_docs = retriever.invoke("What is Ready Tensor all about?")
    
# Print out the retrieved chunks
print("Retrieved Document Chunks:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nChunk {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

Retrieved Document Chunks:

Chunk 1:
Content: ---

Summary: Ready Tensor is a collaborative AI platform emphasizing openness and responsibility. Its Publications System enables sharing of datasets, models, workflows, and research, supported by competitions and robust tools. Security, reproducibility, and community-driven quality assessment are central to its design.
Metadata: {'source': '../documents/ready_tensor_kb_part3.txt'}

Chunk 2:
Content: Ready Tensor Knowledge Base for RAG System  
Comprehensive Overview Based on Documentation

---
Metadata: {'source': '../documents/ready_tensor_kb_part1.txt'}

Chunk 3:
Content: - Pre-trained Models: Domain-specific models (NLP, CV) for transfer learning.
- Learning Materials:
  - Tutorials (e.g., "Deploying a Model via API").
  - Webinars on ethics, reproducibility.
  - Community forums for Q&A.

---
Metadata: {'source': '../documents/ready_tensor_kb_part3.txt'}
